# C17c — WER des phrases camerounaises du test d'OpenSLR 57 (P10, P12, E1)

Le test officiel d'OpenSLR 57 compte 515 phrases de 12 locuteurs (Gabon, Cameroun, Tchad,
Congo). Les 72 phrases camerounaises viennent de 3 locuteurs, tous des hommes. Ce notebook
retranscrit les 515 phrases avec faster-whisper base et small, **sur CPU en int8** (pas besoin
de GPU), puis donne le WER et le CER par pays, avec les mêmes réglages et la même
normalisation (P12) que C17a.

Pourquoi tout le test et pas seulement les 72 phrases : un WER seul ne dit rien sans point de
comparaison. Sur CPU, int8 donne le même WER que float16 sur GPU (C17a : écart non
significatif), et le WER global obtenu ici est comparé à celui de C17a pour le vérifier.

Il faut **Internet activé**. Accélérateur : aucun. Durée : environ 30 minutes.

## a. Démarrage

On clone le dépôt (branche `BRANCH`), on l'installe en mode éditable (I9), puis faster-whisper,
CTranslate2 et jiwer aux versions de C17a.

In [ ]:
import sys

BRANCH = "exp/c17-wer-cameroun"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}
%pip install -q "faster-whisper==1.2.1" "ctranslate2==4.8.5" "jiwer==4.0.0"

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg et OpenSLR 57

`load_audio` décode avec ffmpeg (D1). L'archive officielle (environ 1,8 Go) est décompressée
dans `/tmp`, pour que l'audio ne fasse pas partie des sorties du notebook (I8).

In [ ]:
import shutil
import tarfile
import time
import urllib.request
from pathlib import Path

from echolens.core.data.manifest import dataset_root

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
NOTEBOOK_START = time.time()

OPENSLR57_TAR_URL = "https://openslr.elda.org/resources/57/African_Accented_French.tar.gz"
DOWNLOAD_DIR = Path("/tmp/openslr57")
DATASETS_DIR = Path("/kaggle/working/datasets")

archive_path = DOWNLOAD_DIR / "openslr57.tar.gz"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(OPENSLR57_TAR_URL, archive_path)
with tarfile.open(archive_path) as archive:
    archive.extractall(DOWNLOAD_DIR, filter="data")
archive_path.unlink()
openslr57_root = sorted(DOWNLOAD_DIR.glob("**/African_Accented_French"))[0]

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
link = dataset_root("openslr57", DATASETS_DIR)
if link.is_symlink():
    link.unlink()
link.symlink_to(openslr57_root)
print(link, "→", openslr57_root)

## c. Les phrases du test officiel

Lues dans le manifeste versionné. On vérifie les effectifs : 515 phrases en tout, dont 72
camerounaises de 3 locuteurs.

In [ ]:
from collections import Counter

from echolens.config import REPO_ROOT
from echolens.core.data.manifest import audio_path, load_manifest
from echolens.training.transcription_eval import (
    corpus_rows,
    country_of_speaker,
    load_transcription_eval_config,
)

config = load_transcription_eval_config()
manifest = load_manifest(REPO_ROOT / "data" / "manifests" / "openslr57.csv")
test_selection = None
for selection in config.corpora:
    if selection.id == "test":
        test_selection = selection
test_rows = corpus_rows(manifest, test_selection)

sentences_by_country = Counter(country_of_speaker(row.speaker) for row in test_rows)
cameroon_speakers = {row.speaker for row in test_rows if row.speaker.startswith("cameroon")}
print("Phrases par pays :", dict(sentences_by_country))
print("Locuteurs camerounais :", sorted(cameroon_speakers))
if len(test_rows) != 515 or sentences_by_country["cameroon"] != 72:
    raise RuntimeError("Effectifs inattendus : ne pas continuer.")
missing = [row.path for row in test_rows if not audio_path(row, DATASETS_DIR).exists()]
if missing:
    raise RuntimeError(f"{len(missing)} fichiers absents, par exemple {missing[0]}")

## d. Transcription sur CPU (int8)

Mêmes modèles (révisions figées), même décodage que C17a (`configs/transcription.yaml`). Les
transcriptions brutes sont gardées dans `c17c_hypotheses_int8.json`.

In [ ]:
import json

from echolens.core.audio import load_audio
from echolens.training.transcription_models import load_whisper, transcribe

OUTPUT_DIR = Path("/kaggle/working")
hypotheses = {}
cpu_seconds = {}
for model in config.models:
    whisper = load_whisper(model, "cpu", "int8", config)
    hypotheses[model.id] = {}
    start = time.perf_counter()
    for position, row in enumerate(test_rows):
        signal = load_audio(audio_path(row, DATASETS_DIR))
        hypotheses[model.id][row.path] = transcribe(whisper, signal, config)
        if (position + 1) % 100 == 0:
            print(f"{model.id} : {position + 1} / {len(test_rows)} phrases")
    cpu_seconds[model.id] = round(time.perf_counter() - start, 1)
    del whisper
    print(f"{model.id} : terminé en {cpu_seconds[model.id]:.0f} s")

hypotheses_path = OUTPUT_DIR / "c17c_hypotheses_int8.json"
hypotheses_path.write_text(json.dumps(hypotheses, ensure_ascii=False), encoding="utf-8")

## e. WER et CER par pays

Normalisation P12 des deux côtés, erreurs comptées par jiwer, intervalles à 95 % par bootstrap
sur les locuteurs (comme C17a). **Attention** : avec 3 locuteurs camerounais seulement, leur
intervalle est très large et peu fiable ; il est donné pour l'honnêteté, pas comme preuve.
On contrôle aussi que le WER global int8 est proche de celui de C17a (float16).

In [ ]:
from echolens.training.transcription_eval import (
    bootstrap_intervals,
    error_rates,
    score_sentence,
    scores_by_country,
)

C17A_RESULTS = REPO_ROOT / "training" / "results" / "c17_transcription.json"
c17a = json.loads(C17A_RESULTS.read_text(encoding="utf-8"))


def interval_record(interval):
    return {
        "value": round(interval.value, 4),
        "low": round(interval.low, 4),
        "high": round(interval.high, 4),
    }


def rates_record(scores):
    wer, cer = bootstrap_intervals(
        scores, config.bootstrap_draws, config.seed, config.confidence_level
    )
    speakers = {score.speaker for score in scores}
    return {
        "sentences": len(scores),
        "speakers": len(speakers),
        "wer": interval_record(wer),
        "cer": interval_record(cer),
    }


results = {}
for model in config.models:
    scores = []
    for row in test_rows:
        hypothesis = hypotheses[model.id][row.path]
        scores.append(score_sentence(row.path, row.speaker, row.transcript, hypothesis))
    model_results = {"all_test": rates_record(scores), "by_country": {}}
    grouped = scores_by_country(scores)
    for country in sorted(grouped):
        model_results["by_country"][country] = rates_record(grouped[country])
    # Contrôle : WER global int8 ici contre float16 dans C17a.
    float16_wer = c17a["gpu_float16"][model.id]["test"]["wer"]["value"]
    int8_wer, _ = error_rates(scores)
    model_results["c17a_float16_wer"] = float16_wer
    model_results["int8_minus_c17a_float16"] = round(int8_wer - float16_wer, 4)
    results[model.id] = model_results

for model_id, model_results in results.items():
    print(
        f"=== {model_id} (contrôle : int8 {model_results['all_test']['wer']['value']:.1%}, "
        f"C17a float16 {model_results['c17a_float16_wer']:.1%})"
    )
    for country, record in model_results["by_country"].items():
        wer = record["wer"]
        print(
            f"{country:9s} {record['sentences']:3d} phrases, {record['speakers']} loc. : "
            f"WER {wer['value']:.1%} [{wer['low']:.1%} ; {wer['high']:.1%}]"
        )

## f. Phrases camerounaises suspectes

C17a a repéré dans le test des références suspectes (base **et** small à 100 % de WER ou
plus) : on indique combien sont camerounaises, et le WER camerounais sans elles.

In [ ]:
suspect_paths = {item["path"] for item in c17a["suspect_references"]}
cameroon_suspects = sorted(path for path in suspect_paths if "/cameroon_" in path)
print("Références suspectes camerounaises :", cameroon_suspects)

for model in config.models:
    kept_scores = []
    for row in test_rows:
        if not row.speaker.startswith("cameroon"):
            continue
        if row.path in suspect_paths:
            continue
        hypothesis = hypotheses[model.id][row.path]
        kept_scores.append(score_sentence(row.path, row.speaker, row.transcript, hypothesis))
    record = rates_record(kept_scores)
    results[model.id]["cameroon_without_suspects"] = record
    print(f"{model.id} : WER camerounais sans les suspectes {record['wer']['value']:.1%}")

## g. Résultats

Un seul fichier JSON, à ajouter au dépôt dans `training/results/`.

In [ ]:
output = {
    "protocol": "P10, P12",
    "corpus": "test officiel d'OpenSLR 57, par pays",
    "device": "cpu",
    "compute_type": "int8",
    "models": [model.model_dump() for model in config.models],
    "bootstrap": {
        "draws": config.bootstrap_draws,
        "seed": config.seed,
        "unit": "speaker",
        "confidence_level": config.confidence_level,
    },
    "cameroon_suspect_references": cameroon_suspects,
    "cpu_seconds": cpu_seconds,
    "results": results,
    "runtime_s": round(time.time() - NOTEBOOK_START),
}
result_path = OUTPUT_DIR / "c17c_wer_cameroun.json"
result_path.write_text(json.dumps(output, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(output, indent=2, ensure_ascii=False))

## Nettoyage

On supprime le clone, le lien et l'audio : les sorties ne gardent que les deux JSON (I8).

In [ ]:
shutil.rmtree(CLONE_DIR, ignore_errors=True)
shutil.rmtree(DATASETS_DIR, ignore_errors=True)
shutil.rmtree(DOWNLOAD_DIR, ignore_errors=True)
remaining = [path.name for path in OUTPUT_DIR.iterdir()]
print(sorted(remaining))